# 03 — Exploratory Data Analysis & Statistical Analysis

This notebook builds on the cleaned Olist e-commerce datasets to investigate the main business questions identified during SQL analysis.

## Main questions
1. Is late delivery associated with lower customer satisfaction?
2. Does review score fall as delivery delay becomes more severe?
3. How strong is repeat purchasing?
4. How does commercial performance change over time?

The statistical analysis uses observational data, so results are interpreted as **associations rather than proof of causation**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data/processed")
FIGURE_DIR = Path("../reports/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load processed datasets

Each notebook starts from the saved processed CSV files so the workflow remains reproducible and does not depend on variables created in earlier notebooks.

In [ ]:
customers = pd.read_csv(DATA_DIR / "customers_clean.csv")
orders = pd.read_csv(DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(DATA_DIR / "order_items_clean.csv")
products = pd.read_csv(DATA_DIR / "products_clean.csv")
sellers = pd.read_csv(DATA_DIR / "sellers_clean.csv")
payments = pd.read_csv(DATA_DIR / "payments_clean.csv")
reviews = pd.read_csv(DATA_DIR / "reviews_order_clean.csv")
category_translation = pd.read_csv(DATA_DIR / "category_translation_clean.csv")
geolocation = pd.read_csv(DATA_DIR / "geolocation_zip_clean.csv")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_cols] = orders[date_cols].apply(pd.to_datetime, errors="coerce")

print("Datasets loaded successfully.")

## 2. Build one-row-per-order analysis dataset

`order_items` contains multiple rows for some orders. To avoid duplicating orders during delivery and customer-satisfaction analysis, item-level financial information is aggregated before joining.

In [ ]:
order_financials = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_financials["total_order_value"] = (
    order_financials["product_value"] + order_financials["freight_value"]
)

analysis = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(reviews, on="order_id", how="left")
    .merge(order_financials, on="order_id", how="left")
)

analysis["delivery_days"] = (
    analysis["order_delivered_customer_date"]
    - analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Match SQL DATEDIFF logic by comparing calendar dates, not exact timestamps.
analysis["delay_days"] = (
    analysis["order_delivered_customer_date"].dt.normalize()
    - analysis["order_estimated_delivery_date"].dt.normalize()
).dt.days

analysis["is_late"] = analysis["delay_days"] > 0
analysis["low_review"] = analysis["review_score_mean"] <= 2
analysis["purchase_month"] = (
    analysis["order_purchase_timestamp"].dt.to_period("M").astype(str)
)

delivered = analysis[
    (analysis["order_status"] == "delivered")
    & analysis["order_delivered_customer_date"].notna()
].copy()

print("Order-level analysis rows:", len(analysis))
print("Usable delivered orders:", len(delivered))

## 3. Delivery performance vs customer satisfaction

The first analysis compares review scores for late deliveries against orders delivered on or before the estimated date.

In [ ]:
delivery_review_summary = (
    delivered
    .dropna(subset=["review_score_mean"])
    .groupby("is_late")
    .agg(
        orders=("order_id", "count"),
        avg_review=("review_score_mean", "mean"),
        median_review=("review_score_mean", "median"),
        low_review_rate=("low_review", "mean")
    )
)

delivery_review_summary["low_review_rate"] *= 100
delivery_review_summary.index = ["On time / early", "Late"]

delivery_review_summary.round(3)

In [ ]:
plot_data = delivery_review_summary["avg_review"]

plt.figure(figsize=(7, 5))
plot_data.plot(kind="bar")
plt.title("Average Review Score by Delivery Performance")
plt.xlabel("")
plt.ylabel("Average Review Score")
plt.ylim(0, 5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "review_score_by_delivery_performance.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

## 4. Mann–Whitney U test

**Question:** Do late orders receive systematically different review scores from on-time/early orders?

The Mann–Whitney U test is used because review scores are bounded/ordinal and do not need to satisfy a normality assumption.

In [ ]:
on_time_reviews = delivered.loc[
    (~delivered["is_late"]) & delivered["review_score_mean"].notna(),
    "review_score_mean"
]

late_reviews = delivered.loc[
    delivered["is_late"] & delivered["review_score_mean"].notna(),
    "review_score_mean"
]

u_stat, mw_p_value = mannwhitneyu(
    on_time_reviews,
    late_reviews,
    alternative="two-sided"
)

print("Mann–Whitney U:", u_stat)
print("p-value:", mw_p_value)
print("On-time median:", on_time_reviews.median())
print("Late median:", late_reviews.median())

## 5. Chi-square test: late delivery vs low review

A low review is defined as an order-level review score of 1 or 2. The chi-square test checks whether low-review frequency is independent of delivery status.

In [ ]:
review_delivery_table = pd.crosstab(
    delivered["is_late"],
    delivered["low_review"]
)

review_delivery_table

In [ ]:
chi2, chi_p_value, dof, expected = chi2_contingency(review_delivery_table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", chi_p_value)

## 6. Spearman correlation: delay severity vs review score

Spearman correlation measures the monotonic association between the number of days late/early and customer review score.

In [ ]:
delay_review = delivered.dropna(
    subset=["delay_days", "review_score_mean"]
)

rho, spearman_p_value = spearmanr(
    delay_review["delay_days"],
    delay_review["review_score_mean"]
)

print("Spearman rho:", rho)
print("p-value:", spearman_p_value)

## 7. Review score by delivery-delay band

Delivery performance is grouped into practical business bands so the severity of lateness can be compared with average satisfaction.

In [ ]:
bins = [-np.inf, -7, -1, 0, 3, 7, np.inf]

labels = [
    "7+ days early",
    "1–6 days early",
    "On estimated date",
    "1–3 days late",
    "4–7 days late",
    "8+ days late"
]

delivered["delay_band"] = pd.cut(
    delivered["delay_days"],
    bins=bins,
    labels=labels
)

delay_summary = (
    delivered
    .groupby("delay_band", observed=True)
    .agg(
        orders=("order_id", "count"),
        avg_review=("review_score_mean", "mean")
    )
    .reset_index()
)

delay_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    delay_summary["delay_band"].astype(str),
    delay_summary["avg_review"],
    marker="o"
)
plt.title("Customer Satisfaction Falls as Delivery Delay Increases")
plt.xlabel("Delivery Performance")
plt.ylabel("Average Review Score")
plt.ylim(0, 5)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "review_score_by_delay_band.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

## 8. Customer retention

`customer_unique_id` is used because the same real customer may appear under different order-level `customer_id` values.

In [ ]:
customer_orders = (
    analysis[analysis["order_status"] == "delivered"]
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

customer_type = pd.Series(
    np.where(customer_orders > 1, "Repeat", "One-time"),
    index=customer_orders.index,
    name="customer_type"
)

retention_summary = pd.DataFrame({
    "customers": customer_type.value_counts(),
    "percentage": customer_type.value_counts(normalize=True) * 100
})

retention_summary.round(2)

In [ ]:
plt.figure(figsize=(6, 5))
retention_summary["percentage"].plot(kind="bar")
plt.title("Customer Retention: One-time vs Repeat Customers")
plt.xlabel("")
plt.ylabel("Customers (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "customer_retention.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

## 9. Monthly commercial trend

Only delivered orders are used. The earliest and latest periods should be interpreted carefully because the dataset does not represent complete calendar years at both boundaries.

In [ ]:
monthly = (
    analysis[analysis["order_status"] == "delivered"]
    .groupby("purchase_month")
    .agg(
        orders=("order_id", "nunique"),
        product_value=("product_value", "sum"),
        freight_value=("freight_value", "sum"),
        total_order_value=("total_order_value", "sum")
    )
    .reset_index()
    .sort_values("purchase_month")
)

monthly

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly["purchase_month"], monthly["total_order_value"])
plt.title("Monthly Delivered Order Value")
plt.xlabel("Month")
plt.ylabel("Order Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "monthly_order_value.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

## 10. Statistical results summary

The p-values quantify evidence of association, but they do not establish that delivery delay *causes* lower reviews.

In [ ]:
statistical_results = pd.DataFrame({
    "test": [
        "Mann–Whitney U: late vs on-time review scores",
        "Chi-square: delivery status vs low review",
        "Spearman: delay days vs review score"
    ],
    "statistic": [u_stat, chi2, rho],
    "p_value": [mw_p_value, chi_p_value, spearman_p_value]
})

statistical_results